# Real-Page Extraction Review

Use this notebook to review private/local scanned pages without committing them. It loads `samples/manifest.local.json` when present, otherwise it uses `samples/manifest.example.json`. Missing image files are skipped gracefully.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import cv2
import matplotlib.pyplot as plt

from utils.image.layout.analyzer import LayoutAnalyzer
from utils.image.ops.extract import save_region_crops
from utils.text.text_utils import TextCleaner
from martial_arts_ocr.pipeline.extraction_adapters import (
    image_region_from_extraction,
    page_result_from_extractions,
    text_region_from_extraction,
)

LOCAL_MANIFEST = PROJECT_ROOT / "samples" / "manifest.local.json"
EXAMPLE_MANIFEST = PROJECT_ROOT / "samples" / "manifest.example.json"
MANIFEST_PATH = LOCAL_MANIFEST if LOCAL_MANIFEST.exists() else EXAMPLE_MANIFEST
OUTPUT_ROOT = PROJECT_ROOT / "notebooks" / "output" / "real_page_review"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Using manifest: {MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
samples = manifest.get("samples", [])
print(f"Loaded {len(samples)} sample definitions")

## Classical CV Image Region Review

This section uses the current `LayoutAnalyzer` with YOLO disabled. It writes crops and converts crop metadata into canonical `ImageRegion` objects for review.

In [ ]:
def resolve_sample_path(sample):
    raw_path = sample.get("path", "")
    path = Path(raw_path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def review_image_regions(sample):
    sample_id = sample.get("id", "unnamed_sample")
    image_path = resolve_sample_path(sample)
    print(f"\n=== {sample_id} ===")
    print(sample.get("description", ""))
    print(f"Image path: {image_path}")

    if not image_path.exists():
        print("Skipping image review: file is missing. Add private scans locally under samples/private/.")
        return []

    image = cv2.imread(str(image_path))
    if image is None:
        print("Skipping image review: OpenCV could not read this file.")
        return []

    analyzer = LayoutAnalyzer({
        "use_yolo_figure": False,
        "enabled_detectors": ["figure", "contours"],
        "contours_always": True,
        "filter_text_like": True,
    })
    regions = analyzer.detect_image_regions(image)
    crop_dir = OUTPUT_ROOT / sample_id
    crop_records = save_region_crops(image, regions, crop_dir, prefix="image_region")
    canonical_regions = [
        image_region_from_extraction(record, index=index)
        for index, record in enumerate(crop_records, start=1)
    ]

    expected = sample.get("expected", {})
    print(f"Detected image regions: {len(regions)}")
    print(f"Expected minimum: {expected.get('min_image_regions', 'n/a')}")
    print("Crop records:")
    for record in crop_records:
        print(record)

    overlay = analyzer.debug_draw_regions(image, regions)
    plt.figure(figsize=(8, 10))
    plt.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    plt.title(sample_id)
    plt.axis("off")
    plt.show()

    print("Canonical ImageRegion payloads:")
    for region in canonical_regions:
        print(region.to_dict())
    return crop_records


image_outputs = {sample.get("id", "unnamed_sample"): review_image_regions(sample) for sample in samples}

## Text Cleanup Review

Provide OCR-like text in the manifest as `sample_text` or `ocr_text`. This lets cleanup behavior be reviewed without requiring OCR binaries.

In [ ]:
cleaner = TextCleaner()

def review_text_cleanup(sample):
    sample_id = sample.get("id", "unnamed_sample")
    raw_text = sample.get("sample_text") or sample.get("ocr_text") or ""
    if not raw_text.strip():
        print(f"\n=== {sample_id} text ===")
        print("Skipping text review: no sample_text or ocr_text in manifest.")
        return None

    cleaned, stats = cleaner.clean_text(raw_text, aggressive=False)
    region = text_region_from_extraction({
        "region_id": f"{sample_id}_manual_text",
        "text": cleaned,
        "language": "mixed",
        "confidence": 1.0,
    })

    print(f"\n=== {sample_id} text ===")
    print("Before:\n", raw_text)
    print("After:\n", cleaned)
    print("Stats:", stats.to_dict())
    print("Canonical TextRegion:", region.to_dict())

    expected = sample.get("expected", {})
    notes = expected.get("notes", [])
    if notes:
        print("Review notes:")
        for note in notes:
            print("-", note)
    return region


text_outputs = {sample.get("id", "unnamed_sample"): review_text_cleanup(sample) for sample in samples}

## Canonical PageResult Preview

This preview shows how reviewed utility outputs can map toward canonical pipeline models later, without wiring them into runtime yet.

In [ ]:
for sample in samples:
    sample_id = sample.get("id", "unnamed_sample")
    text_items = []
    if text_outputs.get(sample_id) is not None:
        text_items.append({"region_id": f"{sample_id}_text", "text": text_outputs[sample_id].text})
    image_items = image_outputs.get(sample_id, [])
    page = page_result_from_extractions(
        page_number=1,
        raw_text=text_outputs[sample_id].text if text_outputs.get(sample_id) is not None else "",
        text_items=text_items,
        image_items=image_items,
    )
    print(f"\n=== {sample_id} PageResult ===")
    print(page.to_dict())

## Optional YOLO Check

YOLO remains opt-in. This section only reports availability unless an explicit local model path is added to a sample as `yolo_model_path`.

In [ ]:
from utils.image.layout.detectors.yolo_figure import YOLOFigureDetector

print("ultralytics available:", YOLOFigureDetector.available)
for sample in samples:
    model_path = sample.get("yolo_model_path")
    if not YOLOFigureDetector.available:
        print(f"{sample.get('id')}: skipping YOLO; ultralytics unavailable")
    elif not model_path:
        print(f"{sample.get('id')}: skipping YOLO; no yolo_model_path configured")
    elif not Path(model_path).exists():
        print(f"{sample.get('id')}: skipping YOLO; model file missing: {model_path}")
    else:
        print(f"{sample.get('id')}: local YOLO model configured; instantiate manually for review")